# 03 — Modelling (RQ1)

Answer RQ1: do Random Forest and XGBoost improve classification accuracy over logistic
regression, using only the Four Factors features? Train three models, evaluate them on a
held-out test set, and produce one comparison table.

**Input:** `data/processed/team_games_features.csv`
**Output:** `data/processed/03_model_results.csv`


## Setup and load

Import the libraries and load the feature matrix from notebook 02.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "team_games_features.csv", parse_dates=["Date"])

## Feature set

RQ1 is restricted to the Four Factors. Use only the 12 core features
(`home/away/diff` for eFG, TOV%, ORB%, FTrate). The pace and rest challengers are left out so
this stays a clean "four factors only" baseline. `Date` and `era` are kept for the split, not
as features.


In [2]:
# The 12 core Four Factors features.
FACTORS = ["eFG", "TOV_pct", "ORB_pct", "FTrate"]
FEATURES = []
for f in FACTORS:
    FEATURES += [f"home_{f}", f"away_{f}", f"diff_{f}"]

## Train/test split

Chronological split with no shuffle. Within each era the games are sorted by date and the last
20% become the test set, the first 80% the train set. Splitting inside each era keeps temporal
order (no future leakage) and keeps both eras present in the test set.


In [3]:
# Last 20% of each era (by date) is the test set; the first 80% is train.
train_parts = []
test_parts = []
for era, group in df.groupby("era"):
    group = group.sort_values("Date")
    cut = int(len(group) * 0.8)
    train_parts.append(group.iloc[:cut])
    test_parts.append(group.iloc[cut:])

train = pd.concat(train_parts)
test = pd.concat(test_parts)

X_train, y_train = train[FEATURES], train["y"]
X_test, y_test = test[FEATURES], test["y"]

## Models

Three classifiers with simple default settings (no tuning yet). Logistic regression is scaled
with `StandardScaler` inside a pipeline (fit on train only); the tree models use raw features.


In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, random_state=42),
}

## Cross-validation and test evaluation

For each model: 5-fold cross-validation on the training set with `shuffle=False` (keeps
temporal order) for a mean CV accuracy, then fit on the full training set and score on the
held-out test set with accuracy, F1, and ROC-AUC.


In [5]:
cv = KFold(n_splits=5, shuffle=False)
results = []

for name, model in models.items():
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean()

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "cv_accuracy": cv_acc,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1": f1_score(y_test, pred),
        "test_roc_auc": roc_auc_score(y_test, proba),
    })

## Results table

Collect the scores into one table (one row per model), show it, and save it.


In [6]:
results_table = pd.DataFrame(results).round(4)
results_table.to_csv(PROCESSED_DIR / "03_model_results.csv", index=False)
results_table

,model,cv_accuracy,test_accuracy,test_f1,test_roc_auc
0,Logistic Regression,0.6227,0.6274,0.7139,0.6602
1,Random Forest,0.6039,0.6112,0.6908,0.6320
2,XGBoost,0.5775,0.5792,0.6536,0.5813
